In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import glob

from scripts.constants import corr_strings

In [2]:
files = sorted(glob.glob("./data/muT_V15/results_*.json"))

dfs = [pd.read_json(f) for f in files]

df = pd.concat(dfs, ignore_index=True)

In [3]:
df.head(10)

,mu,T,best_free,free,F_onsite,F_swave,F_dwave,F_px,F_py,Fuu_px,Fuu_py,Fdd_px,Fdd_py
0,0.081633,0.010000,-1538.789922,-1538.789922,0,0,"{'imag': -0.1026150047, 'real': 0.0}",0,0,0,0,0,0
1,0.081633,0.010000,-1538.789922,-1538.789922,0,0,"{'imag': 0.1026150047, 'real': 0.0}",0,0,0,0,0,0
2,0.081633,0.010000,-1538.789922,-1538.789922,0,0,"{'imag': -0.0796178144, 'real': -0.0647367192}",0,0,0,0,0,0
3,0.081633,0.010000,-1538.789922,-1538.788568,0,0,"{'imag': 0.0, 'real': 0.10261453720000001}",0,0,0,0,0,0
4,0.081633,0.010000,-1538.789922,-1538.789922,0,0,"{'imag': 3.8330660199999997e-16, 'real': -0.10...",0,0,0,0,0,0
5,0.081633,0.015918,-1538.790290,-1538.790290,0,0,"{'imag': -0.1026141787, 'real': 0.0}",0,0,0,0,0,0
6,0.081633,0.015918,-1538.790290,-1538.790290,0,0,"{'imag': 0.1026141787, 'real': 0.0}",0,0,0,0,0,0
7,0.081633,0.015918,-1538.790290,-1538.789375,0,0,"{'imag': -0.07961692840000001, 'real': -0.0647...",0,0,0,0,0,0
8,0.081633,0.015918,-1538.790290,-1538.789772,0,0,"{'imag': 0.0, 'real': 0.1026139999}",0,0,0,0,0,0
9,0.081633,0.015918,-1538.790290,-1538.789375,0,0,"{'imag': 3.8957385169999997e-16, 'real': -0.10...",0,0,0,0,0,0


In [4]:
df.count()

mu           5529
T            5529
best_free    5529
free         5529
F_onsite     5529
F_swave      5529
F_dwave      5529
F_px         5529
F_py         5529
Fuu_px       5529
Fuu_py       5529
Fdd_px       5529
Fdd_py       5529
dtype: int64

In [5]:
import pandas as pd
import numpy as np
import ast

# all F_* columns, in their existing left-to-right order
F_cols = [c for c in df.columns if c.startswith("F_")]

def get_real(x):
    """
    Return the real part of an F_* entry.
    Handles:
      - plain numbers
      - complex numbers
      - dicts like {'real': ..., 'imag': ...}
      - strings representing such dicts
    """
    if pd.isna(x):
        return -np.inf

    if isinstance(x, dict):
        return float(x.get("real", -np.inf))

    if isinstance(x, complex):
        return float(x.real)

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)

    if isinstance(x, str):
        s = x.strip()
        try:
            y = ast.literal_eval(s)
            if isinstance(y, dict):
                return float(y.get("real", -np.inf))
            if isinstance(y, complex):
                return float(y.real)
            if isinstance(y, (int, float)):
                return float(y)
        except Exception:
            pass

    return -np.inf

F_real = df[F_cols].applymap(get_real)
sort_df = pd.concat(
    [df[['mu', 'T']], F_real.add_suffix('_real')],
    axis=1
)

sort_cols = ['mu', 'T'] + list(F_real.add_suffix('_real').columns)
ascending = [True, True] + [False] * len(F_real.columns)

df1 = (
    df.loc[
        sort_df.sort_values(sort_cols, ascending=ascending, kind='mergesort').index
    ]
    .drop_duplicates(subset=['mu', 'T'], keep='first')
    .reset_index(drop=True)
)

df1.iloc[700:720]

C:\Users\emoeu\AppData\Local\Temp\ipykernel_8624\695141412.py:44: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  F_real = df[F_cols].applymap(get_real)


,mu,T,best_free,free,F_onsite,F_swave,F_dwave,F_px,F_py,Fuu_px,Fuu_py,Fdd_px,Fdd_py
700,1.22449,0.010000,-1744.256875,-1744.256875,0,0,0,"{'imag': 5.286332624e-18, 'real': 0.0328465096...","{'imag': 0.032846509600000004, 'real': -7.8492...",0,0,0,0
701,1.22449,0.011837,-1744.256113,-1744.256113,0,0,0,"{'imag': 7.755433665e-18, 'real': 0.0328442211}","{'imag': 0.0328442211, 'real': -7.591109273000...",0,0,0,0
702,1.22449,0.013673,-1744.253675,-1744.253675,0,0,0,"{'imag': 7.023808957e-18, 'real': 0.0328370038}","{'imag': 0.0328370038, 'real': -6.505001277e-18}",0,0,0,0
703,1.22449,0.015510,-1744.248040,-1744.248040,0,0,0,"{'imag': 1.1102484360000001e-17, 'real': 0.032...","{'imag': 0.0328203122, 'real': -6.339829852000...",0,0,0,0
704,1.22449,0.017347,-1744.237387,-1744.237387,0,0,0,"{'imag': 6.304466226e-18, 'real': 0.0327886052}","{'imag': 0.0327886052, 'real': -8.005520143e-18}",0,0,0,0
705,1.22449,0.019184,-1744.220247,-1744.220247,0,0,0,"{'imag': 6.682878196e-18, 'real': 0.0327368004}","{'imag': 0.0327368004, 'real': -6.569799297000...",0,0,0,0
706,1.22449,0.021020,-1744.194156,-1744.194156,0,0,0,"{'imag': 6.5244830340000006e-18, 'real': 0.032...","{'imag': 0.0326577339, 'real': -4.724538022e-18}",0,0,0,0
707,1.22449,0.022857,-1744.158237,-1744.158237,0,0,0,"{'imag': 4.586683409e-19, 'real': 0.0325473675}","{'imag': 0.0325473675, 'real': -3.277593989000...",0,0,0,0
708,1.22449,0.024694,-1744.110815,-1744.110815,0,0,0,"{'imag': 9.16722583e-18, 'real': 0.03240006}","{'imag': 0.03240006, 'real': -2.176874674e-18}",0,0,0,0
709,1.22449,0.026531,-1744.050764,-1744.050764,0,0,0,"{'imag': 1.005957504e-17, 'real': 0.0322112194...","{'imag': 0.032211219400000005, 'real': -8.4902...",0,0,0,0


In [7]:
import pandas as pd
import numpy as np
import ast

F_cols = [c for c in df.columns if c.startswith("F_")]

def to_complex(x):
    """
    Convert an entry to a complex number.
    Handles:
      - dicts like {'real': ..., 'imag': ...}
      - strings representing such dicts
      - python complex
      - real numbers
    """
    if pd.isna(x):
        return 0.0 + 0.0j

    if isinstance(x, dict):
        return complex(float(x.get("real", 0.0)), float(x.get("imag", 0.0)))

    if isinstance(x, complex):
        return x

    if isinstance(x, (int, float, np.integer, np.floating)):
        return complex(float(x), 0.0)

    if isinstance(x, str):
        s = x.strip()
        try:
            y = ast.literal_eval(s)
            if isinstance(y, dict):
                return complex(float(y.get("real", 0.0)), float(y.get("imag", 0.0)))
            if isinstance(y, complex):
                return y
            if isinstance(y, (int, float)):
                return complex(float(y), 0.0)
        except Exception:
            pass

    raise ValueError(f"Cannot parse value {x!r} as complex.")

def canonical_phase_vector(row, cols, tol=1e-10, decimals=12):
    """
    Return a canonical representation of the F_* vector modulo global phase.

    Two rows are considered equivalent iff their canonical vectors match.
    """
    v = np.array([to_complex(row[c]) for c in cols], dtype=np.complex128)

    mags = np.abs(v)
    nz = np.where(mags > tol)[0]

    # completely zero vector
    if len(nz) == 0:
        return tuple((0.0, 0.0) for _ in cols)

    # fix global phase using the leftmost nonzero component
    k = nz[0]
    phase = np.exp(-1j * np.angle(v[k]))
    v = v * phase

    # remove tiny floating noise
    v.real[np.abs(v.real) < tol] = 0.0
    v.imag[np.abs(v.imag) < tol] = 0.0

    # force anchor entry to be positive real
    if v[k].real < 0:
        v = -v
        v.real[np.abs(v.real) < tol] = 0.0
        v.imag[np.abs(v.imag) < tol] = 0.0

    # rounded tuple so it can be hashed / compared
    return tuple((round(z.real, decimals), round(z.imag, decimals)) for z in v)

def sort_key_largest_F_real(row, cols):
    """
    Sorting key:
      - larger largest real(F_*) first
      - ties broken by the leftmost column attaining that value
    """
    reals = [to_complex(row[c]).real for c in cols]
    return tuple(reals)   # lexicographic, left-to-right

# build canonical phase signature
df2 = df.copy()
df2["_phase_sig"] = df2.apply(lambda r: canonical_phase_vector(r, F_cols), axis=1)

# optional: sorting within each (mu,T) block before deduplication
# this preserves your earlier preference:
#   sort by mu, then T, then by real parts of F_* left-to-right descending
sort_aux = df2[F_cols].applymap(lambda x: to_complex(x).real)
sort_aux.columns = [f"{c}__real" for c in F_cols]

df2 = pd.concat([df2, sort_aux], axis=1)

sort_cols = ["mu", "T"] + list(sort_aux.columns)
ascending = [True, True] + [False] * len(sort_aux.columns)

df2 = df2.sort_values(sort_cols, ascending=ascending, kind="mergesort")

# now drop only phase-equivalent duplicates within each (mu,T)
df2 = (
    df2.drop_duplicates(subset=["mu", "T", "_phase_sig"], keep="first")
       .drop(columns=["_phase_sig"] + list(sort_aux.columns))
       .reset_index(drop=True)
)
df2.iloc[50:100]

C:\Users\emoeu\AppData\Local\Temp\ipykernel_8624\2937976830.py:92: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  sort_aux = df2[F_cols].applymap(lambda x: to_complex(x).real)


,mu,T,best_free,free,F_onsite,F_swave,F_dwave,F_px,F_py,Fuu_px,Fuu_py,Fdd_px,Fdd_py
50,0.081633,0.140204,-1534.541763,-1534.541763,0,0,"{'imag': -0.07427050860000001, 'real': -0.0603...",0,0,0,0,0,0
51,0.081633,0.146122,-1533.869275,-1533.869275,0,0,"{'imag': 0.0, 'real': 0.0946092526}",0,0,0,0,0,0
52,0.081633,0.146122,-1533.869275,-1533.869275,0,0,"{'imag': -0.0734062424, 'real': -0.05968613110...",0,0,0,0,0,0
53,0.081633,0.152041,-1533.138856,-1533.138856,0,0,"{'imag': 0.0, 'real': 0.0933760149}",0,0,0,0,0,0
54,0.081633,0.152041,-1533.138856,-1533.138856,0,0,"{'imag': -0.07244938740000001, 'real': -0.0589...",0,0,0,0,0,0
55,0.081633,0.157959,-1532.351331,-1532.351331,0,0,"{'imag': 0.0, 'real': 0.09201664300000001}",0,0,0,0,0,0
56,0.081633,0.157959,-1532.351331,-1532.351331,0,0,"{'imag': -0.0713946661, 'real': -0.05805053170...",0,0,0,0,0,0
57,0.081633,0.163878,-1531.505083,-1531.505083,0,0,"{'imag': 0.0, 'real': 0.0905222109}",0,0,0,0,0,0
58,0.081633,0.163878,-1531.505083,-1531.505083,0,0,"{'imag': -0.0702351533, 'real': -0.0571077394}",0,0,0,0,0,0
59,0.081633,0.169796,-1530.601557,-1530.601557,0,0,"{'imag': 0.0, 'real': 0.0888844808}",0,0,0,0,0,0
